<a href="https://colab.research.google.com/github/Pensive1881/DSR44_2025/blob/main/%5Breplicator_cones_spheres_cubes%5Dvisualize_lidar_and_rgb_with_fiftyone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizing RGB + LiDAR Data with FiftyOne

### Antonio Rueda-Toicen
### antonio@kineto.ai


This notebook demonstrates how to visualize co-registered RGB images and LiDAR point clouds using FiftyOne's grouped dataset feature.

## Overview

FiftyOne supports multimodal visualization through grouped datasets, allowing you to:
- View RGB images and point clouds side-by-side
- Switch between modalities using the group slices dropdown
- Synchronize navigation across different data types
- Add labels and metadata for analysis




## Image and LiDAR Sensor Data

https://drive.google.com/drive/folders/1sPoBLVY-ho4IolgCzszGU6xnz4uPW6Mu?usp=drive_link

Follow the [instructions here](https://github.com/andandandand/practical-computer-vision/blob/main/docs/add_shortcut_to_google_drive.md) to link the data to your Google Drive.


In [1]:
%%capture
!uv pip install fiftyone==1.10.0

In [2]:
import fiftyone as fo
import numpy as np
from pathlib import Path
from tqdm import tqdm
import os
import pandas as pd


/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


In [3]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


In [4]:
data_path = Path('/gdrive/MyDrive/multimodal_training_workshop/data')
os.listdir(data_path)

['flower_photos.gsheet',
 'cubes_only_single_mode_results.gsheet',
 'flower_photos',
 'replicator_data_cubes',
 'assessment',
 'replicator_data_orthogonal',
 'replicator_data_orthogonal.gz',
 'incomplete_replicator_data_orthogonal',
 'dup_replicator_data_orthogonal',
 'rgb_best.pth']

## Configuration

Set the paths to your dataset. We'll use the `replicator_data_orthogonal` dataset by default.

In [5]:
# Dataset configuration
FIFTYONE_DATASET_NAME = "replicator_shapes_multimodal"
DATASET_PATH = data_path / "replicator_data_orthogonal"

os.listdir(DATASET_PATH)

['azimuth (1).npy',
 'colors.csv',
 'azimuth.npy',
 'positions.csv',
 'zenith.npy',
 'pointcloud',
 'distance',
 'lidar',
 'rgb',
 'colors.gsheet',
 'pcd',
 'pcd.zip']

In [6]:
# Paths to different modalities
RGB_DIR = DATASET_PATH / "rgb"
PCD_DIR = DATASET_PATH / "pcd"


In [7]:
POSITIONS_DF = pd.read_csv("https://raw.githubusercontent.com/andandandand/practical-computer-vision/refs/heads/main/artifacts/replicator_data_orthogonal_positions.csv")
COLORS_DF = pd.read_csv("https://raw.githubusercontent.com/andandandand/practical-computer-vision/refs/heads/main/artifacts/replicator_data_orthogonal_colors.csv")

In [8]:
POSITIONS_DF.head()

,cube_x,cube_y,cube_z,sphere_x,sphere_y,sphere_z,torus_x,torus_y,torus_z
0,-0.059180,-0.762632,4.637073,2.172168,-4.251295,0.112740,-1.016680,2.577287,-0.659820
1,3.852501,2.610654,1.582966,-3.358410,4.437363,-4.714490,1.354207,-3.176319,-1.989396
2,-0.586377,-0.825559,4.393541,-1.258265,1.063964,2.084718,0.209829,-0.261561,-4.296398
3,2.541017,-2.623700,-3.518759,-0.025759,4.889721,-3.133865,2.022122,1.653047,3.616594
4,1.187262,-1.649648,-1.238787,-2.808753,2.319092,-0.563041,3.399651,0.271180,1.989490


In [9]:
# How many samples to load (set to None for all)
MAX_SAMPLES = 100  # Start with 100 samples for faster initial loading

print(f"Dataset path: {DATASET_PATH}")
print(f"RGB directory: {RGB_DIR}")
print(f"PCD directory: {PCD_DIR}")
print(f"Loading up to {MAX_SAMPLES if MAX_SAMPLES else 'all'} samples")

Dataset path: /gdrive/MyDrive/multimodal_training_workshop/data/replicator_data_orthogonal
RGB directory: /gdrive/MyDrive/multimodal_training_workshop/data/replicator_data_orthogonal/rgb
PCD directory: /gdrive/MyDrive/multimodal_training_workshop/data/replicator_data_orthogonal/pcd
Loading up to 100 samples


In [10]:
len(os.listdir(RGB_DIR))

9999

## Verify Data Files

In [11]:
# Check if directories exist
assert RGB_DIR.exists(), f"RGB directory not found: {RGB_DIR}"
assert PCD_DIR.exists(), f"PCD directory not found: {PCD_DIR}. Run convert_npy_to_pcd.py first!"

# Count files
rgb_files = sorted(RGB_DIR.glob("*.png"))
pcd_files = sorted(PCD_DIR.glob("*.pcd"))

print(f"Found {len(rgb_files)} RGB images")
print(f"Found {len(pcd_files)} PCD point clouds")

# Verify matching files
rgb_stems = {f.stem for f in rgb_files}
pcd_stems = {f.stem for f in pcd_files}
matching = rgb_stems & pcd_stems

print(f"Matching pairs: {len(matching)}")

if len(matching) == 0:
    print("\n⚠️  ERROR: No matching RGB/PCD pairs found!")
    print("Please run convert_npy_to_pcd.py first to generate .pcd files.")
else:
    print(f"\n✅ Ready to create dataset with {len(matching)} samples")

Found 9999 RGB images
Found 9999 PCD point clouds
Matching pairs: 9999

✅ Ready to create dataset with 9999 samples


## Load Metadata

Load position and color information for each sample.

In [12]:
# Load positions (x, y, z for 3 objects = 9 values per sample)
positions = POSITIONS_DF.values
print(f"Loaded positions from DataFrame: {positions.shape}")
print(f"Sample positions (file 0): {positions[0]}")

# Load colors (if available in cubes dataset)
colors = COLORS_DF.astype(str).values
print(f"\nLoaded colors from DataFrame: {colors.shape}")
print(f"Sample colors (file 0): {colors[0]}")


Loaded positions from DataFrame: (9999, 9)
Sample positions (file 0): [-0.05917953 -0.76263237  4.63707304  2.17216849 -4.25129461  0.11274032
 -1.01668012  2.57728744 -0.65981996]

Loaded colors from DataFrame: (9999, 9)
Sample colors (file 0): ['0.8869178891181946' '0.2956545948982239' '0.5414272546768188'
 '0.7731012105941772' '0.7087929248809814' '0.9369980692863464'
 '0.0277276560664176' '0.4278150498867035' '0.0352732017636299']


## Create FiftyOne Grouped Dataset

A grouped dataset allows us to associate RGB images with their corresponding point clouds.

In [13]:
# Delete existing dataset if it exists
if FIFTYONE_DATASET_NAME in fo.list_datasets():
    print(f"Deleting existing dataset: {FIFTYONE_DATASET_NAME}")
    fo.delete_dataset(FIFTYONE_DATASET_NAME)

# Create new grouped dataset
print(f"Creating new dataset: {FIFTYONE_DATASET_NAME}")
dataset = fo.Dataset(FIFTYONE_DATASET_NAME, persistent=True)
dataset.add_group_field("group", default="rgb")

print(f"✅ Created grouped dataset: {FIFTYONE_DATASET_NAME}")

Creating new dataset: replicator_shapes_multimodal
✅ Created grouped dataset: replicator_shapes_multimodal


## Add Samples to Dataset

For each matching RGB/PCD pair, we create a group with two slices:
- `rgb`: The camera image
- `lidar`: The point cloud

In [14]:
# Get matching file pairs
file_stems = sorted(matching)

# Limit samples if specified
if MAX_SAMPLES:
    file_stems = file_stems[:MAX_SAMPLES]

print(f"Adding {len(file_stems)} grouped samples...")

samples = []
for stem in tqdm(file_stems, desc="Creating samples"):
    # Get file paths
    rgb_path = str(RGB_DIR / f"{stem}.png")
    pcd_path = str(PCD_DIR / f"{stem}.pcd")

    # Parse file index
    file_idx = int(stem)

    # Create group
    group = fo.Group()

    # Create RGB sample
    rgb_sample = fo.Sample(
        filepath=rgb_path,
        group=group.element("rgb")
    )

    # Create PCD sample
    pcd_sample = fo.Sample(
        filepath=pcd_path,
        group=group.element("lidar")
    )

    # Add metadata to both samples
    if positions is not None and file_idx < len(positions):
        pos = positions[file_idx]

        # Store positions as custom fields
        metadata = {
            "file_index": file_idx,
            "cube1_x": float(pos[0]),
            "cube1_y": float(pos[1]),
            "cube1_z": float(pos[2]),
            "cube2_x": float(pos[3]),
            "cube2_y": float(pos[4]),
            "cube2_z": float(pos[5]),
            "cube3_x": float(pos[6]),
            "cube3_y": float(pos[7]),
            "cube3_z": float(pos[8]),
        }

        # Add metadata to both samples
        for key, value in metadata.items():
            rgb_sample[key] = value
            pcd_sample[key] = value

    # Add color information if available
    if colors is not None and file_idx < len(colors):
        color_row = colors[file_idx]
        rgb_sample["cube1_color"] = str(color_row[0])
        rgb_sample["cube2_color"] = str(color_row[1])
        rgb_sample["cube3_color"] = str(color_row[2])
        pcd_sample["cube1_color"] = str(color_row[0])
        pcd_sample["cube2_color"] = str(color_row[1])
        pcd_sample["cube3_color"] = str(color_row[2])

    samples.extend([rgb_sample, pcd_sample])

# Add all samples to dataset
dataset.add_samples(samples)

print(f"\n✅ Added {len(file_stems)} grouped samples to dataset")
print(f"Total samples in dataset: {len(dataset)}")
print(f"Group slices: {dataset.group_slices}")

Adding 100 grouped samples...


Creating samples: 100%|██████████| 100/100 [00:00<00:00, 2126.07it/s]


 100% |█████████████████| 200/200 [287.1ms elapsed, 0s remaining, 703.2 samples/s]  


INFO:eta.core.utils: 100% |█████████████████| 200/200 [287.1ms elapsed, 0s remaining, 703.2 samples/s]  



✅ Added 100 grouped samples to dataset
Total samples in dataset: 100
Group slices: ['rgb', 'lidar']


## Dataset Statistics

In [15]:
print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Dataset name: {dataset.name}")
print(f"Total samples: {len(dataset)}")
print(f"Group field: {dataset.group_field}")
print(f"Group slices: {dataset.group_slices}")
print(f"Media types: {dataset.group_media_types}")
print(f"\nSample fields:")
for field_name, field in dataset.get_field_schema().items():
    print(f"  - {field_name}: {field}")
print("="*60)


DATASET SUMMARY
Dataset name: replicator_shapes_multimodal
Total samples: 100
Group field: group
Group slices: ['rgb', 'lidar']
Media types: {'rgb': 'image', 'lidar': 'point-cloud'}

Sample fields:
  - id: fiftyone.core.fields.ObjectIdField
  - filepath: fiftyone.core.fields.StringField
  - tags: fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
  - metadata: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.Metadata)
  - created_at: fiftyone.core.fields.DateTimeField
  - last_modified_at: fiftyone.core.fields.DateTimeField
  - group: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.groups.Group)
  - file_index: fiftyone.core.fields.IntField
  - cube1_x: fiftyone.core.fields.FloatField
  - cube1_y: fiftyone.core.fields.FloatField
  - cube1_z: fiftyone.core.fields.FloatField
  - cube2_x: fiftyone.core.fields.FloatField
  - cube2_y: fiftyone.core.fields.FloatField
  - cube2_z: fiftyone.core.fields.FloatField
  - cube3_x: fiftyone.core.fields.FloatF

## Launch FiftyOne App

This will open the FiftyOne App in your browser where you can:
- View RGB images and point clouds side-by-side
- Use the group slices dropdown to switch between modalities
- Filter samples by metadata (positions, colors)
- Navigate through the dataset interactively

In [16]:
# Launch the app
session = fo.launch_app(dataset, auto=False)
print(session.url)
print("\n" + "="*60)
print("FiftyOne App launched!")
print("="*60)
print("\nUsage tips:")
print("1. Use the 'group slices' dropdown to switch between RGB and LiDAR views")
print("2. Click on samples to view them in detail")
print("3. Use the sidebar to filter by metadata (positions, colors)")
print("4. For point clouds, use mouse to rotate, zoom, and pan")
print("\nPress Ctrl+C in the terminal to stop the session when done.")
print("="*60)

Session launched. Run `session.show()` to open the App in a cell output.


INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.



Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.10.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



INFO:fiftyone.core.session.session:
Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.10.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



https://5151-m-s-2g7qhbfgenk2u-a.asia-east1-0.prod.colab.dev?polling=true

FiftyOne App launched!

Usage tips:
1. Use the 'group slices' dropdown to switch between RGB and LiDAR views
2. Click on samples to view them in detail
3. Use the sidebar to filter by metadata (positions, colors)
4. For point clouds, use mouse to rotate, zoom, and pan

Press Ctrl+C in the terminal to stop the session when done.


## Example Queries

You can filter and query the dataset programmatically:

In [17]:
# Example: Get only RGB slice
rgb_view = dataset.select_group_slices("rgb")
print(f"RGB samples: {len(rgb_view)}")

# Example: Get only LiDAR slice
lidar_view = dataset.select_group_slices("lidar")
print(f"LiDAR samples: {len(lidar_view)}")

# Example: Filter by position (find samples where cube1 is on the right side, x > 2)
if "cube1_x" in dataset.get_field_schema():
    filtered_view = dataset.match(fo.ViewField("cube1_x") > 2.0)
    print(f"Samples with cube1_x > 2.0: {len(filtered_view)}")

# Example: Filter by color (if available)
if "cube1_color" in dataset.get_field_schema():
    red_cubes = dataset.match(fo.ViewField("cube1_color") == "red")
    print(f"Samples with red cube1: {len(red_cubes)}")

RGB samples: 100
LiDAR samples: 100
Samples with cube1_x > 2.0: 38
Samples with red cube1: 0


## Visualize Specific Samples

You can also load specific samples by index:

In [18]:
# View a specific sample (e.g., file 163 from the notebook examples)
specific_view = dataset.match(fo.ViewField("file_index") == 163)

if len(specific_view) > 0:
    print(f"Found sample 163")
    session.view = specific_view
else:
    print("Sample 163 not found (may not be in the first 100 samples)")

Sample 163 not found (may not be in the first 100 samples)


## Load More Samples

If you want to load all 9,999 samples, run this cell (warning: may take a few minutes):

In [19]:
# Uncomment to load all samples
# MAX_SAMPLES = None
# # Re-run the "Add Samples to Dataset" cell above to load all samples

## Cleanup

To delete the dataset and free up space:

In [20]:
# Uncomment to delete the dataset
# fo.delete_dataset(DATASET_NAME)
# print(f"Deleted dataset: {DATASET_NAME}")

## Next Steps

- Explore the FiftyOne documentation: https://docs.voxel51.com/
- Add custom labels or annotations to samples
- Export subsets for training
- Integrate with model predictions
- Create custom views and filters

## References

- [FiftyOne Grouped Datasets](https://docs.voxel51.com/user_guide/groups.html)
- [3D Visualization](https://docs.voxel51.com/user_guide/app.html#3d-visualizer)
- [Point Cloud Support](https://docs.voxel51.com/user_guide/app.html#point-clouds)